# Dartboard Retrieval

Pick results that jointly cover the query and each other - relevance *and* diversity in one score.

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

Dartboard (simplified): think of throwing darts at the query. A good *set* of results is one where, wherever the dart lands, some selected passage is close. Greedy selection maximises the Gaussian-smoothed coverage of the candidate pool.

In [ ]:
import numpy as np

def dartboard(query, k=4, pool=20, sigma=0.4, div_weight=1.0, rel_weight=1.0):
    q = embed(query)[0]
    ids = [h["idx"] for h in store.search(query, pool)]
    V = store.vecs[ids]
    dist = 1 - V @ V.T                         # pairwise cosine distance
    qdist = 1 - V @ q
    log_p = -(dist ** 2) / (2 * sigma ** 2)    # log of Gaussian "closeness"
    chosen = []
    best_cover = np.full(len(ids), -np.inf)    # best log-prob of each candidate being covered by a chosen doc
    for _ in range(k):
        scores = []
        for i in range(len(ids)):
            if i in chosen:
                scores.append(-np.inf); continue
            cover = np.maximum(best_cover, log_p[i])
            scores.append(rel_weight * (-(qdist[i] ** 2) / (2 * sigma ** 2)) + div_weight * np.logaddexp.reduce(cover))
        pick = int(np.argmax(scores))
        chosen.append(pick)
        best_cover = np.maximum(best_cover, log_p[pick])
    return [{"idx": ids[i], "text": chunks[ids[i]], "score": float(1 - qdist[i]), "meta": {}} for i in chosen]

query = "How can I make a RAG system more accurate?"
print("--- top-k"); show(store.search(query, 4), width=90)
print("--- dartboard"); show(dartboard(query), width=90)

In [ ]:
print(answer(query, [h["text"] for h in dartboard(query)]))

Tune `div_weight` up for broader coverage, down for a tight match to the query.